In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

device = torch.device("cpu")
torch.set_num_threads(4)  # i5 de 11ª geração: 4 núcleos físicos (teste 4 e 8 depois)

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

PyTorch: 2.15.0+cpu
Dispositivo: cpu


In [2]:
with open("../dados/dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Caracteres muito raros (emojis, símbolos soltos) só aumentam o vocabulário
contagem = Counter(text)
text = "".join(c if contagem[c] >= 5 else "?" for c in text)

print("Número de caracteres:", len(text))
print()
print(text[:500])

Número de caracteres: 2693666

@@@ ARQUIVO: spring-data-examples/jpa/showcase/src/test/java/example/springdata/jpa/showcase/after/AccountRepositoryIntegrationTest.java @@@
package example.springdata.jpa.showcase.after;

import static org.assertj.core.api.Assertions.*;

import example.springdata.jpa.showcase.core.Account;

import java.util.Collections;

import org.junit.jupiter.api.Test;

import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.boot.test.context.SpringBootTest;
import org.sprin


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

print("|V| =", vocab_size)
print([repr(c) for c in chars])

|V| = 97
["'\\n'", "' '", "'!'", '\'"\'', "'#'", "'$'", "'%'", "'&'", '"\'"', "'('", "')'", "'*'", "'+'", "','", "'-'", "'.'", "'/'", "'0'", "'1'", "'2'", "'3'", "'4'", "'5'", "'6'", "'7'", "'8'", "'9'", "':'", "';'", "'<'", "'='", "'>'", "'?'", "'@'", "'A'", "'B'", "'C'", "'D'", "'E'", "'F'", "'G'", "'H'", "'I'", "'J'", "'K'", "'L'", "'M'", "'N'", "'O'", "'P'", "'Q'", "'R'", "'S'", "'T'", "'U'", "'V'", "'W'", "'X'", "'Y'", "'Z'", "'['", "'\\\\'", "']'", "'_'", "'`'", "'a'", "'b'", "'c'", "'d'", "'e'", "'f'", "'g'", "'h'", "'i'", "'j'", "'k'", "'l'", "'m'", "'n'", "'o'", "'p'", "'q'", "'r'", "'s'", "'t'", "'u'", "'v'", "'w'", "'x'", "'y'", "'z'", "'{'", "'|'", "'}'", "'~'", "'©'", "'ü'"]


In [4]:
exemplo = '@RestController\npublic class Foo {\n    <div class="flex">'
codificado = encode(exemplo)
print("Tokens:", codificado[:20], "...")
print("Reconstrução igual ao original?", decode(codificado) == exemplo)

Tokens: [33, 51, 69, 83, 84, 36, 79, 78, 84, 82, 79, 76, 76, 69, 82, 0, 80, 85, 66, 76] ...
Reconstrução igual ao original? True


In [5]:
import re

partes = [p for p in re.split(r"(?m)^(?=@@@ ARQUIVO: )", text) if p.strip()]

por_repo = {}
for p in partes:
    nome = re.match(r"@@@ ARQUIVO: (.+?) @@@", p).group(1)
    repo = nome.split("/")[0]
    por_repo.setdefault(repo, []).append(p)

total = len(text)
for repo, ps in sorted(por_repo.items(), key=lambda kv: -sum(map(len, kv[1]))):
    n = sum(map(len, ps))
    print(f"{n:>9,} chars | {n/total:5.1%} | {len(ps):>5} arquivos | {repo}")

1,033,671 chars | 38.4% |   722 arquivos | spring-data-examples
  907,860 chars | 33.7% |   256 arquivos | tailwind-starter-kit
  326,581 chars | 12.1% |    10 arquivos | windmill-dashboard
  170,336 chars |  6.3% |    69 arquivos | spring-petclinic
  113,528 chars |  4.2% |    42 arquivos | flowbite
  103,839 chars |  3.9% |    78 arquivos | spring-petclinic-microservices
   26,194 chars |  1.0% |     1 arquivos | Admin-Template
   11,657 chars |  0.4% |     4 arquivos | pagila


In [6]:
import re

cabecalhos = re.findall(r"@@@ ARQUIVO: (.+?) @@@", text)
grupos = Counter("/".join(c.split("/")[:3]) for c in cabecalhos)

for grupo, qtd in grupos.most_common(8):
    print(f"{qtd:5d}  {grupo}")

   59  spring-data-examples/jpa/example
   49  spring-petclinic/src/main
   45  spring-data-examples/jdbc/howto
   45  spring-data-examples/cassandra/example
   38  spring-data-examples/mongodb/example
   24  flowbite/layouts/_shortcodes
   22  spring-data-examples/jpa/showcase
   21  spring-data-examples/jpa/multitenant


In [7]:
VAL_REPOS = {"spring-petclinic", "spring-petclinic-microservices", "windmill-dashboard"}

train_text = "".join(p for repo, ps in por_repo.items() if repo not in VAL_REPOS for p in ps)
val_text   = "".join(p for repo, ps in por_repo.items() if repo in VAL_REPOS for p in ps)

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data   = torch.tensor(encode(val_text), dtype=torch.long)

total = len(train_data) + len(val_data)
print(f"treino: {len(train_data):,} | validação: {len(val_data):,} ({len(val_data)/total:.1%})")

treino: 2,092,910 | validação: 600,756 (22.3%)


In [8]:
import math, time, csv, os

torch.manual_seed(1337)

d = 128           # dimensão do embedding
T = 256           # janela de contexto
L = 4             # blocos Transformer
h = 4             # cabeças de atenção
batch_size = 32

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(0, len(source) - T - 1, (batch_size,))
    X = torch.stack([source[i:i+T] for i in starts])
    Y = torch.stack([source[i+1:i+T+1] for i in starts])
    return X, Y

X, Y = get_batch("train")
print("X:", X.shape, "| Y:", Y.shape)   # esperado: (32, 128)

X: torch.Size([32, 256]) | Y: torch.Size([32, 256])


In [9]:
class Head(nn.Module):
    def __init__(self, d, head_dim, T):
        super().__init__()
        self.key = nn.Linear(d, head_dim, bias=False)
        self.query = nn.Linear(d, head_dim, bias=False)
        self.value = nn.Linear(d, head_dim, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))
        self.head_dim = head_dim

    def forward(self, x):                                   # x: (B, T, d)
        Q, K, V = self.query(x), self.key(x), self.value(x)  # cada um: (B, T, head_dim)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)  # (B, T, T)
        Tc = x.size(1)
        scores = scores.masked_fill(self.mask[:Tc, :Tc] == 0, float("-inf"))
        A = F.softmax(scores, dim=-1)                       # (B, T, T)
        return A @ V                                        # (B, T, head_dim)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        assert d % h == 0
        self.heads = nn.ModuleList([Head(d, d // h, T) for _ in range(h)])
        self.projection = nn.Linear(d, d)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)  # (B, T, d)
        return self.projection(out)


class FeedForward(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d, h, T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attention = MultiHeadAttention(d, h, T)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = FeedForward(d)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d, T, L, h):
        super().__init__()
        self.T = T
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, h, T) for _ in range(L)])
        self.ln_final = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab_size)

    def forward(self, idx):                                  # idx: (B, T)
        B, Tc = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(Tc, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_final(x))                # (B, T, vocab_size)

In [10]:
model = MiniGPT(vocab_size, d, T, L, h).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("Parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")

X, Y = get_batch("train")
logits = model(X)
loss = loss_fn(logits.reshape(-1, vocab_size), Y.reshape(-1))
print("Loss inicial:", round(loss.item(), 3), "| esperado ≈", round(math.log(vocab_size), 3))

Parâmetros: 849,505
Loss inicial: 4.715 | esperado ≈ 4.575


In [11]:
from torch.optim.lr_scheduler import CosineAnnealingLR

os.makedirs("../logs", exist_ok=True)
os.makedirs("../modelos", exist_ok=True)

max_steps = 4000
eval_every = 250
scheduler = CosineAnnealingLR(optimizer, T_max=max_steps)

@torch.no_grad()
def estimate_loss(n=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        vals = []
        for _ in range(n):
            X, Y = get_batch(split)
            vals.append(loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1)).item())
        out[split] = sum(vals) / n
    model.train()
    return out

best_val = float("inf")

with open("../logs/treino_v3.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "train_loss", "val_loss", "lr", "d", "T", "L", "h", "batch_size"])
    t0 = time.time()
    for step in range(max_steps + 1):
        if step % eval_every == 0:
            r = estimate_loss()
            lr_atual = optimizer.param_groups[0]["lr"]
            writer.writerow([step, round(r["train"], 4), round(r["val"], 4),
                             round(lr_atual, 6), d, T, L, h, batch_size])
            f.flush()
            marca = ""
            if r["val"] < best_val:
                best_val = r["val"]
                torch.save({"model": model.state_dict(), "chars": chars,
                            "hparams": dict(d=d, T=T, L=L, h=h)}, "../modelos/minigpt_v3.pt")
                marca = " *salvo*"
            print(f"step {step:4d} | train {r['train']:.3f} | val {r['val']:.3f} | "
                  f"{(time.time() - t0) / 60:.1f} min{marca}")
        X, Y = get_batch("train")
        loss = loss_fn(model(X).reshape(-1, vocab_size), Y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

print("Fim. Melhor val loss:", round(best_val, 3))

step    0 | train 4.722 | val 4.795 | 0.2 min *salvo*
step  250 | train 2.291 | val 2.245 | 3.2 min *salvo*
step  500 | train 1.653 | val 1.926 | 6.4 min *salvo*
step  750 | train 1.248 | val 1.641 | 9.5 min *salvo*
step 1000 | train 1.016 | val 1.522 | 12.6 min *salvo*
step 1250 | train 0.914 | val 1.441 | 15.6 min *salvo*
step 1500 | train 0.842 | val 1.411 | 18.7 min *salvo*
step 1750 | train 0.772 | val 1.417 | 22.2 min
step 2000 | train 0.750 | val 1.345 | 25.8 min *salvo*
step 2250 | train 0.688 | val 1.400 | 29.0 min
step 2500 | train 0.617 | val 1.383 | 32.0 min
step 2750 | train 0.607 | val 1.377 | 35.3 min
step 3000 | train 0.611 | val 1.313 | 38.6 min *salvo*
step 3250 | train 0.591 | val 1.368 | 42.0 min
step 3500 | train 0.597 | val 1.368 | 45.0 min
step 3750 | train 0.579 | val 1.350 | 47.9 min
step 4000 | train 0.583 | val 1.363 | 50.8 min
Fim. Melhor val loss: 1.313


In [12]:
for arq in ["../logs/treino_v2.csv", "../logs/treino_v3.csv"]:
    with open(arq) as f:
        linhas = list(csv.reader(f))
    print(arq)
    for r in linhas[1:][::4] + linhas[-1:]:
        print(f"  step {r[0]:>5} | train {r[1]} | val {r[2]}")

../logs/treino_v2.csv
  step     0 | train 4.7204 | val 4.7072
  step  1000 | train 1.0419 | val 1.5132
  step  2000 | train 0.7598 | val 1.4675
  step  3000 | train 0.6247 | val 1.3159
  step  4000 | train 0.62 | val 1.4039
  step  5000 | train 0.5347 | val 1.3255
  step  6000 | train 0.5424 | val 1.3432
  step  6000 | train 0.5424 | val 1.3432
../logs/treino_v3.csv
  step     0 | train 4.7219 | val 4.7949
  step  1000 | train 1.0163 | val 1.5216
  step  2000 | train 0.75 | val 1.3449
  step  3000 | train 0.6111 | val 1.3128
  step  4000 | train 0.5834 | val 1.3633
  step  4000 | train 0.5834 | val 1.3633


In [13]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.8, top_k=20):
    for _ in range(max_new_tokens):
        logits = model(idx[:, -model.T:])[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

def carregar(caminho, T_):
    m = MiniGPT(vocab_size, d, T_, L, h).to(device)
    m.load_state_dict(torch.load(caminho)["model"])
    m.eval()
    return m

def completar_com(m, prompt, n=400, **kw):
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    return decode(generate(m, idx, n, **kw)[0].tolist())

modelos = {"v2 (T=128)": carregar("../modelos/minigpt_v2.pt", 128),
           "v3 (T=256)": carregar("../modelos/minigpt_v3.pt", 256)}

for nome, m in modelos.items():
    for p in ["@RestController\npublic class", '<div class="flex']:
        torch.manual_seed(42)
        print("=" * 60)
        print(nome, "|", repr(p))
        print(completar_com(m, p, temperature=0.6))

v2 (T=128) | '@RestController\npublic class'
@RestController
public class CategoryIgetName() {

         assertThat(repository.findAll();
                });
                                                   .expectNextCount("Walter", "Busine", "Cater", 43));
                                                         </td>
                                                                                                                                                   .e
v2 (T=128) | '<div class="flex'
<div class="flex items-center justify-between px-heading pl-0 peopace-ss troke-white p-1" data-dui-tab-target="alert" class="text-stone-800 flex items-center justify-center border align-middle select-none font-sans font-medium text-center duration-300 ease-in disabled:opacity-50 disabled:shadow-none disabled:cursor-not-allowed focus:shadow-none text-sm py-2 px-4 shadow-sm hover:shadow-md bg-stone-800 hover:bg-sto
v3 (T=256) | '@RestController\npublic class'
@RestController
public class Cat

In [ ]:
def taxa_copia(amostra, janela=40):
    trechos = [amostra[i:i+janela] for i in range(0, len(amostra) - janela, 10)]
    achados = sum(1 for t in trechos if t in train_text)
    return achados / max(1, len(trechos))

for p in ["@RestController\npublic class", "@Entity\npublic class", '<div class="flex']:
    g = completar(p, n=600)
    print(repr(p), "->", f"{taxa_copia(g):.0%} dos trechos de 40 caracteres existem no dataset")

'@RestController\npublic class' -> 2% dos trechos de 40 caracteres existem no dataset
'@Entity\npublic class' -> 9% dos trechos de 40 caracteres existem no dataset
'<div class="flex' -> 62% dos trechos de 40 caracteres existem no dataset


In [21]:
for temp in [0.5, 0.8, 1.1]:
    print(f"--- temperatura {temp} ---")
    print(completar("@RestController\npublic class", n=300, temperature=temp))

--- temperatura 0.5 ---
@RestController
public class ReactiveAirlineGates {

    private @Autowired CustomerRepository repository;

    @Autowired ReactiveMongoTemplate template;

    private final @Size("Dave") is person. Long country a {

         this.template.context.save(new Comment("Mattthews"));
        var user = repository.save(new Customer(
--- temperatura 0.8 ---
@RestController
public class MongoTenanyToProviduction types {

    private @Id UID id Map<String, String> response;

    private String username;
    public List<User> lastname;

    @Bean
    Request persons before presenting to contention.</small>
     </div>
   <div class="flex h-screen gap-1 sm:flow-md bg-opacit-70">
    
--- temperatura 1.1 ---
@RestController
public class EmployeeRepository extends Meterner(@Nav-schriph"symalil(result.hasRelateBuffer)//
              .addReplicated<Customer, Long> createAuthorsNumber(OfAuthStream a runing a through
           MA meta mavmin ling every from.
        to o appn 

In [14]:
m = modelos["v3 (T=256)"]
torch.manual_seed(42)
saida = completar_com(m, "@RestController\npublic class", n=600, temperature=0.6)
print(saida.split("@@@ FIM @@@")[0])

@RestController
public class CategorySet(classes = new Contact() {
          repository.deleteAll();
     }

    public void setPerson() {
        return false;
    }

    public Long get() {
        return gid;
    }
   public void setAddress() {
         return name;
    }
}

